# Lekcija 14: Nelinearni filtri

Svi dosadašnji filtri, poput filtra srednje vrijednosti, Gaussova, Sobelova i Laplacianova filtra, bili su konvolucije: fiksne linearne kombinacije vrijednosti susjednih piksela. Međutim, linearni filtri imaju ograničenja &mdash; ne razlikuju susjedne piksele koji su „izdvojene vrijednosti” od onih koji su „s druge strane ruba”. Ova lekcija obrađuje filtre koji narušavaju tu linearnost: **medijansko filtriranje**, **eroziju i dilataciju slike u sivim tonovima kao filtre minimuma i maksimuma** te **bilateralni filtar**, koji čuva rubove.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Medijansko filtriranje: otpornost na izdvojene vrijednosti

Medijanski filtar zamjenjuje svaki piksel **medijanom**, a ne srednjom vrijednošću, njegova susjedstva. Dok jedna izrazito pogrešna vrijednost piksela može drastično utjecati na prosjek, medijan mijenja vrlo malo ili nimalo &mdash; zato je medijansko filtriranje posebno učinkovito protiv **šuma tipa sol i papar**, odnosno izoliranih piksela nasumično postavljenih na crnu ili bijelu vrijednost, s kojim se linearno zaglađivanje loše nosi.


In [ ]:
clean = np.zeros((120, 120), dtype=np.uint8)
cv2.rectangle(clean, (25, 25), (95, 95), 200, -1)

rng = np.random.default_rng(2)
salt_pepper = clean.copy()
hit = rng.random(clean.shape) < 0.06
salt_pepper[hit & (rng.random(clean.shape) < 0.5)] = 0
salt_pepper[hit & (rng.random(clean.shape) >= 0.5)] = 255

median_result = cv2.medianBlur(salt_pepper, 5)
gaussian_result = cv2.GaussianBlur(salt_pepper, (5, 5), sigmaX=1.5)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [salt_pepper, gaussian_result, median_result],
                          ['Salt-and-pepper noise', 'Gaussian blur', 'Median filter']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('mean abs error vs. clean image:')
print(f'  Gaussian blur : {np.abs(gaussian_result.astype(int) - clean.astype(int)).mean():.2f}')
print(f'  Median filter : {np.abs(median_result.astype(int) - clean.astype(int)).mean():.2f}')

Gaussovo zamućivanje razmazuje ekstremnu vrijednost svakog oštećenog piksela na susjede, ostavljajući slabe točkaste tragove posvuda. Medijanski filtar jednostavno nadglasava izolirani loš piksel njegovim brojnim ispravnim susjedima, uklanjajući gotovo sav šum i zadržavajući oštre rubove.


## Filtri minimuma i maksimuma: erozija i dilatacija u sivim tonovima

U Lekciji 3 uveli smo eroziju i dilataciju na *binarnim* slikama. Na slici **u sivim tonovima** prirodno se generaliziraju: erozija zamjenjuje svaki piksel **najmanjom** vrijednošću njegova susjedstva, a dilatacija **najvećom**. Obje su nelinearne, odnosno nisu ponderirani zbroj, i obje čuvaju rubove drukčije od zamućivanja &mdash; pomiču ih umjesto da ih zamućuju.


In [ ]:
def min_filter(image, ksize):
    r = ksize // 2
    padded = cv2.copyMakeBorder(image, r, r, r, r, cv2.BORDER_REPLICATE)
    out = np.zeros_like(image)
    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            out[i, j] = padded[i:i + ksize, j:j + ksize].min()
    return out

small = clean[::4, ::4]  # smaller image so the manual pixel loop stays fast
mine = min_filter(small, 3)
reference = cv2.erode(small, np.ones((3, 3), np.uint8))
print('manual min filter == cv2.erode?', np.array_equal(mine, reference))

## Problem koji linearno zamućivanje ne rješava: zaglađivanje bez prelaska preko rubova

Gaussovo zamućivanje tretira susjede jednako s obzirom na razliku njihovih vrijednosti &mdash; upravo zato zamućuje preko rubova. **Bilateralni filtar** to rješava ponderiranjem svakog susjeda prema *dvama* čimbenicima: prostornoj blizini, kao kod Gaussova zamućivanja, **i** sličnosti njegova intenziteta intenzitetu središnjeg piksela:

$$I'(p) = \frac{1}{W_p}\sum_{q \in N(p)} G_{\sigma_s}(\|p - q\|) \cdot G_{\sigma_r}(|I(p) - I(q)|) \cdot I(q)$$

gdje $W_p$ normalizira težine tako da im zbroj bude 1. Prostorno bliski susjedi *sličnog* intenziteta dobivaju veliku težinu i zajedno se zaglađuju; prostorno bliski susjedi vrlo različitog intenziteta, odnosno oni s druge strane ruba, dobivaju težinu gotovo nula, pa rub ostaje sačuvan.


In [ ]:
def bilateral_filter(image, radius, sigma_spatial, sigma_range):
    image_f = image.astype(np.float64)
    ys, xs = np.mgrid[-radius:radius + 1, -radius:radius + 1]
    spatial_weight = np.exp(-(xs**2 + ys**2) / (2 * sigma_spatial**2))
    padded = cv2.copyMakeBorder(image_f, radius, radius, radius, radius, cv2.BORDER_REFLECT101)

    out = np.zeros_like(image_f)
    h, w = image.shape
    for i in range(h):
        for j in range(w):
            patch = padded[i:i + 2 * radius + 1, j:j + 2 * radius + 1]
            range_weight = np.exp(-(patch - image_f[i, j])**2 / (2 * sigma_range**2))
            weight = spatial_weight * range_weight
            out[i, j] = (weight * patch).sum() / weight.sum()
    return out

### Provjera usporedbom s bibliotekom OpenCV

Funkcija `cv2.bilateralFilter` koristi istu ideju uz interne aproksimacije radi brzine, pa očekujemo blisko podudaranje, ali ne i potpuno jednake vrijednosti svakog piksela.


In [ ]:
step = np.zeros((40, 40), dtype=np.uint8)
step[20:, :] = 200
noisy_step = np.clip(step.astype(np.float64) + rng.normal(0, 10, step.shape), 0, 255).astype(np.uint8)

mine = bilateral_filter(noisy_step, radius=3, sigma_spatial=3, sigma_range=25)
cv_result = cv2.bilateralFilter(noisy_step, d=7, sigmaColor=25, sigmaSpace=3).astype(np.float64)

print(f'max abs difference  = {np.abs(mine - cv_result).max():.2f} gray levels')
print(f'mean abs difference = {np.abs(mine - cv_result).mean():.2f} gray levels')

### Usporedni prikaz očuvanja rubova

Ključno pitanje usporedbe jest: zaglađuje li filtar šum na šumnom skokovitom rubu uz očuvanje njegove oštrine ili zajedno sa šumom zamućuje i rub?


In [ ]:
big_step = np.zeros((150, 150), dtype=np.uint8)
big_step[75:, :] = 200
noisy_big = np.clip(big_step.astype(np.float64) + rng.normal(0, 15, big_step.shape), 0, 255).astype(np.uint8)

gaussian_smoothed = cv2.GaussianBlur(noisy_big, (0, 0), sigmaX=4)
bilateral_smoothed = cv2.bilateralFilter(noisy_big, d=9, sigmaColor=40, sigmaSpace=15)

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for ax, im, title in zip(axes[0], [noisy_big, gaussian_smoothed, bilateral_smoothed],
                          ['Noisy step edge', 'Gaussian blur', 'Bilateral filter']):
    ax.imshow(im, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title, fontsize=9)
    ax.axis('off')

col = 75
for ax, im, title in zip(axes[1], [noisy_big, gaussian_smoothed, bilateral_smoothed],
                          ['profile: noisy', 'profile: Gaussian', 'profile: bilateral']):
    ax.plot(im[:, col])
    ax.set_ylim(0, 255)
    ax.set_title(title, fontsize=9)
plt.tight_layout()
plt.show()

Profil nakon Gaussova zamućivanja pokazuje postupan prijelaz preko mnogo redaka &mdash; rub je vidljivo omekšan. Profil nakon bilateralnog filtriranja ostaje jednolik sa svake strane, jer je šum uklonjen, ali i dalje naglo skače u blizini retka 75 &mdash; rub je sačuvan zato što su pikseli s različitih strana granice bili previše različiti po intenzitetu da bi se zajedno usrednjavali, bez obzira na prostornu blizinu.


### Zadaci

1. Primijenite medijanski filtar umjesto Gaussova ili bilateralnog na `noisy_big` i dodajte njegov profil usporedbi. Kako se u očuvanju ruba uspoređuje s bilateralnim filtrom?
2. U funkciji `bilateral_filter` postavite `sigma_range` na vrlo veliku vrijednost, primjerice 1000. Na koji bi se postupak filtar tada trebao svesti i potvrđuje li to vaš rezultat?
3. Bilateralno filtriranje znatno je sporije od Gaussova zamućivanja jer se težine moraju ponovno izračunavati u svakom pikselu: ovise o lokalnom intenzitetu, a ne samo o položaju. Usporedite vlastitu funkciju `bilateral_filter` s funkcijom `cv2.bilateralFilter` na slici veličine 100 × 100 pomoću `%timeit` te riječima opišite što implementacija biblioteke OpenCV vjerojatno radi drukčije da bi bila brža.
